# Madrid · renta vs coste de vivienda por barrios

**Pregunta:** ¿en qué barrios y distritos de Madrid se vive mejor con el sueldo y en cuáles vas más apurado, cruzando la **renta** con el **coste de la vivienda** (alquiler y compra)?

**Indicadores:**
- **Esfuerzo de alquiler (%)** = parte de la renta anual del hogar que se va en alquilar un piso tipo (70 m²). <30% holgado · 30-40% ajustado · >40% apurado.
- **Esfuerzo de compra (años)** = años de renta íntegra del hogar para comprar una vivienda tipo (90 m²).

**Fuentes (datos reales):** renta → INE *Atlas de Distribución de Renta de los Hogares*; vivienda → Ayto. Madrid / Mº Vivienda; geometría → Ayto. Madrid.

> ⚠️ **Aviso:** por defecto este notebook usa **datos SINTÉTICOS** reproducibles (semilla fija) para que funcione sin descargar nada. No son cifras oficiales. Para datos reales, ejecuta `src/extract.py` en tu máquina (con red) y cambia `SOURCE = 'real'`. Ver el README del proyecto.


## 1. Configuración e imports


In [ ]:
import sys, os
# Permite importar los módulos del proyecto desde src/
SRC = os.path.abspath('../src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

import pandas as pd
import matplotlib.pyplot as plt

import config
from synthetic import generar_barrios
from transform import add_indicators, resumen_por_distrito

pd.set_option('display.float_format', lambda x: f'{x:,.1f}')
SOURCE = 'synthetic'   # cambia a 'real' cuando tengas los datos del INE descargados


## 2. Carga de datos

Con `SOURCE='synthetic'` se generan los 131 barrios de forma determinista. Con `'real'`, carga aquí tu dataset descargado y unido (ver `src/extract.py`).


In [ ]:
if SOURCE == 'synthetic':
    base = generar_barrios()
else:
    # TODO datos reales: carga data/processed/madrid_barrios.csv tras ejecutar la extracción
    base = pd.read_csv(config.DATA_PROCESSED / 'madrid_barrios.csv')

df = add_indicators(base)
print(f'{len(df)} barrios · {df.cod_distrito.nunique()} distritos')
df.head()


## 3. ¿Dónde se vive mejor y dónde más apurado? (alquiler)

Ordenamos los barrios por **esfuerzo de alquiler**: cuanto menor, mejor se vive con el sueldo.


In [ ]:
cols = ['barrio','renta_hogar','alquiler_eur_m2_mes','esfuerzo_alquiler_pct','clasificacion_alquiler']
print('TOP 10 — donde MEJOR se vive (menor esfuerzo):')
display(df.nsmallest(10, 'esfuerzo_alquiler_pct')[cols].reset_index(drop=True))
print('\nTOP 10 — donde MÁS apurado se vive (mayor esfuerzo):')
display(df.nlargest(10, 'esfuerzo_alquiler_pct')[cols].reset_index(drop=True))


## 4. Esfuerzo de alquiler por distrito

Agregamos a los 21 distritos (media de sus barrios). Líneas en 30% y 40% (umbrales de asequibilidad).


In [ ]:
r = resumen_por_distrito(df).sort_values('esfuerzo_alquiler_pct')
fig, ax = plt.subplots(figsize=(9,7))
colors = ['#2ca25f' if v<config.UMBRAL_ALQUILER_HOLGADO else '#fec44f' if v<config.UMBRAL_ALQUILER_APURADO else '#de2d26' for v in r['esfuerzo_alquiler_pct']]
ax.barh(r['distrito'], r['esfuerzo_alquiler_pct'], color=colors)
ax.axvline(30, ls='--', c='grey', lw=1); ax.axvline(40, ls='--', c='grey', lw=1)
ax.set_xlabel('% de la renta del hogar en alquiler (piso 70 m²)')
ax.set_title(f'Madrid · esfuerzo de alquiler por distrito  [{SOURCE.upper()}]')
plt.tight_layout(); plt.show()


## 5. La paradoja: renta vs esfuerzo

Cada punto es un barrio. El color indica los años de renta para comprar. Suele verse que **los barrios de menor renta soportan MÁS esfuerzo de alquiler**, aunque paguen menos €/m² en absoluto: los ingresos caen más rápido que el precio.


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sc = ax.scatter(df['renta_hogar'], df['esfuerzo_alquiler_pct'], c=df['esfuerzo_compra_anios'], cmap='viridis', s=30, alpha=0.85)
ax.axhline(40, ls='--', c='red', lw=1, alpha=0.5)
ax.set_xlabel('Renta media del hogar (€/año)')
ax.set_ylabel('Esfuerzo de alquiler (%)')
ax.set_title(f'131 barrios · renta vs esfuerzo  [{SOURCE.upper()}]')
plt.colorbar(sc, label='Años de renta para comprar (90 m²)')
plt.tight_layout(); plt.show()

corr = df['renta_hogar'].corr(df['esfuerzo_alquiler_pct'])
print(f'Correlación renta ↔ esfuerzo de alquiler: {corr:.2f} (negativa = a menos renta, más esfuerzo)')


## 6. Compra de vivienda

Años de renta íntegra del hogar para comprar 90 m².


In [ ]:
rc = resumen_por_distrito(df).sort_values('esfuerzo_compra_anios')
fig, ax = plt.subplots(figsize=(9,7))
ax.barh(rc['distrito'], rc['esfuerzo_compra_anios'], color='#3182bd')
ax.set_xlabel('Años de renta del hogar para comprar (90 m²)')
ax.set_title(f'Madrid · esfuerzo de compra por distrito  [{SOURCE.upper()}]')
plt.tight_layout(); plt.show()


## 7. Mapa por barrios (solo con datos reales)

Para el mapa coroplético necesitas la geometría de los barrios (GeoJSON del Ayto. de Madrid). Con datos sintéticos no hay polígonos reales, así que esta celda está preparada pero desactivada. Con `SOURCE='real'` y el GeoJSON descargado, se activa.


In [ ]:
# Requiere geopandas y el GeoJSON de barrios de Madrid en data/raw/.
# import geopandas as gpd
# barrios = gpd.read_file(config.DATA_RAW / 'madrid_barrios.geojson')
# gdf = barrios.merge(df, on='cod_barrio')
# gdf.plot(column='esfuerzo_alquiler_pct', legend=True, cmap='RdYlGn_r', figsize=(10,10))
#
# Mapa interactivo con Kepler.gl:
# from keplergl import KeplerGl
# m = KeplerGl(height=600); m.add_data(gdf, 'esfuerzo')
# m.save_to_html(file_name=str(config.OUTPUTS / 'mapa_madrid.html'))
print('Mapa disponible al usar datos reales + geometría de barrios.')


## 8. Conclusiones y limitaciones

**Conclusiones (patrón típico, a confirmar con datos reales):**
- El esfuerzo de **alquiler** es el indicador más discriminante entre barrios.
- Los barrios de **menor renta** tienden a soportar **mayor esfuerzo** relativo, pese a pagar menos €/m² en absoluto.
- El esfuerzo de **compra** (años) es más uniforme, pero alto en toda la ciudad.

**Limitaciones:**
- Los datos mostrados son **sintéticos** salvo que `SOURCE='real'`.
- Renta (hogar) y precios de vivienda provienen de fuentes distintas y con metodologías propias.
- La vivienda 'tipo' (70/90 m²) es una convención; cambiarla altera los valores absolutos.
- Falacia ecológica: el dato es por barrio, no por individuo.

**Reproducir:** `python src/build_dataset.py --source synthetic` (o `--source real` tras la extracción).
